# 02 - Build fine-tuning data, arm A (tourism-filtered ILCI + Samanantar)
A T4 GPU runtime makes the LaBSE alignment check fast (CPU also works, slower). Downloads ILCI (62 MB) and Samanantar Hindi (1.9 GB), filters tourism-like pairs,
removes everything that overlaps the test sets, and writes train/dev files + a report.

**Before running:** accept the BPCC terms while logged in to Hugging Face:
https://huggingface.co/datasets/ai4bharat/BPCC

In [7]:
# Cell 1: Hugging Face login (BPCC and FLORES+ are gated)
from huggingface_hub import login, whoami
try:
    print("Logged in as", whoami()["name"])
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face token: "))

Logged in as KiyotakaAyanokoji


In [8]:
# Cell 2: get (or update) the code from GitHub
import os
REPO_URL = "https://github.com/prefierolasoledad/TourMT-Hi.git"
WORKDIR = "/content/tourmt-hi"
if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} reset -q --hard origin/main
%cd {WORKDIR}
!pip install -r requirements.txt 2>&1 | grep -iE "error|failed" || true
!git log -1 --oneline

/content/tourmt-hi
e2da7d2 (HEAD -> main, origin/main, origin/HEAD) Enhance shortlist creation and build fine-tuning data for tourism-like English-Hindi pairs


In [9]:
# Cell 3: build arm A (about 15-25 minutes: downloading + reading ~10M Samanantar pairs, then LaBSE)
!python src/build_finetune_data.py --config configs/finetune_arm_a.yaml

Building test-overlap index...
Resolving data files: 100% 234/234 [00:00<00:00, 20752.90it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 20885.33it/s]
Resolving data files: 100% 234/234 [00:00<00:00, 10256.63it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 12020.47it/s]
Resolving data files: 100% 234/234 [00:00<00:00, 10036.89it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 12082.01it/s]
Resolving data files: 100% 234/234 [00:00<00:00, 21729.74it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 22172.11it/s]
Resolving data files: 100% 234/234 [00:00<00:00, 21689.40it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 21475.11it/s]
Resolving data files: 100% 234/234 [00:00<00:00, 20136.38it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 21136.27it/s]
  12150 test sentences: {'flores_dev_en': 997, 'flores_dev_hi': 997, 'flores_dev_mai': 997, 'flores_devtest_en': 1012, 'flores_devtest_hi': 1012, 'flores_devtest_mai': 1012, 'tourism_candidates_english': 5675, 'to

In [10]:
# Cell 4: show 30 random training pairs so we can judge the quality by eye
import csv, random
rows = list(csv.DictReader(open("data/finetune/arm_a/train.tsv", encoding="utf-8"), delimiter="\t"))
for r in random.Random(0).sample(rows, min(30, len(rows))):
    print(f"[{r['source']}] [LaBSE {r.get('labse', '')}] [{r['keywords']}]\n  EN: {r['en']}\n  HI: {r['hi']}")

[samanantar] [LaBSE ] [museum,souvenir,visitors]
  EN: The inauguration of Phase-II of the Museum completes the Rashtrapati Bhavan Museum Complex, which comprises the Stables Museum (Phase I), the Garages Museum (Phase II) and the Clock Tower, which will be home to the Visitors Reception Centre, Cafeteria and Souvenir shop.
  HI: चरण-2 संग्रहालय का उद्घाटन राष्ट्रपति भवन संग्रहालय परिसर को संपूर्ण करता है, जिसमें अस्तबल संग्रहालय, (चरण-1), गैराज संग्रहालय (चरण-2) और क्लॉक टावर है जिसमें आगंतुकों का स्वागत कक्ष, कैफेटेरिया और स्मारकों की दुकान होगी।
[samanantar] [LaBSE ] [hostel,timings]
  EN: Students wanted to meet the vice-chancellor and demand withdrawal of the draft hostel manual, which they claimed has provisions for the fee hike, dress code and curfew timings.
  HI: छात्र कुलपति से मिलना चाहते थे और उनकी मांग थी कि मसौदा छात्रावास मैनुअल को वापस लिया जाये जिसमें उनके अनुसार फीस वृद्धि, कर्फ्यू का वक्त और ड्रेस कोड जैसी पाबंदियों का प्रावधान है.
[samanantar] [LaBSE ] [hotel,hotels

In [11]:
# Cell 5: put the report into this notebook's output so it can be saved on the laptop (then Ctrl+S)
print("===REPORT-START===")
print(open("results/finetune_data_arm_a.json", encoding="utf-8").read())
print("===REPORT-END===")

===REPORT-START===
{
  "name": "arm_a",
  "config": {
    "name": "arm_a",
    "sources": {
      "ilci": {
        "repo": "ai4bharat/BPCC",
        "file": "ilci/hin_Deva.tsv",
        "revision": "5bde309374f21c39e7c3d505f28ae11f58f2a220"
      },
      "samanantar": {
        "repo": "ai4bharat/samanantar",
        "config": "hi",
        "revision": "ead34c5b22a0354d350e7dfdd6aece7df5a48f37",
        "max_rows": null
      }
    },
    "min_keyword_hits": 2,
    "core_keywords": [
      "hotel",
      "hotels",
      "guest house",
      "guesthouse",
      "hostel",
      "resort",
      "homestay",
      "check-in",
      "temple",
      "temples",
      "mosque",
      "church",
      "gurudwara",
      "monastery",
      "shrine",
      "pilgrim",
      "pilgrims",
      "pilgrimage",
      "ghat",
      "ghats",
      "fort",
      "palace",
      "monument",
      "monuments",
      "museum",
      "heritage",
      "ruins",
      "tomb",
      "mausoleum",
      "tourist",
